# Retos · Matrices y tableros

Retos sobre listas de listas: operaciones de matrices, recorridos en espiral, validadores del estado final de un tablero (cuadrado mágico, sudoku, bingo, tres en raya, hundir la flota), costas de islas, la hormiga de Langton, rejillas hexagonales y algunos problemas de listas con pilas, `Counter` y conjuntos. Cada solución se contrasta con fuerza bruta, con una segunda versión o con NumPy. Ideas de [Edabit](https://edabit.com/es/challenges/python3) y de los [ejercicios de José A. Alonso](https://github.com/jaalonso/Ejercicios-Python).

## 1. Operaciones con matrices como listas

Escribir, para matrices guardadas como listas de listas, `transpuesta`, `producto` (con `ValueError` si las dimensiones no encajan), `rota_horario`, `rota_antihorario` y `potencia(m, k)` por exponenciación rápida (elevar al cuadrado y multiplicar solo cuando el bit de $k$ vale 1). Compararlas con NumPy (`.T`, `@`, `np.rot90` y `np.linalg.matrix_power`) en matrices aleatorias. Calcular después $F_{100}$ con la potencia de $\begin{pmatrix}1&1\\1&0\end{pmatrix}$ y ver que los enteros de Python no se desbordan, mientras que NumPy con `int64` sí lo hace sin avisar.

In [1]:
import random
from collections import Counter, deque
from itertools import chain, groupby, product
from math import comb

import numpy as np


def transpuesta(m):
    return [list(fila) for fila in zip(*m)]


def producto(a, b):
    if len(a[0]) != len(b):
        raise ValueError(f"no se puede multiplicar {len(a)}×{len(a[0])} por {len(b)}×{len(b[0])}")
    columnas = list(zip(*b))
    return [[sum(x * y for x, y in zip(fila, col)) for col in columnas] for fila in a]


def rota_horario(m):
    return [list(fila) for fila in zip(*m[::-1])]


def rota_antihorario(m):
    return [list(fila) for fila in zip(*m)][::-1]


def identidad(n):
    return [[int(i == j) for j in range(n)] for i in range(n)]


def potencia(m, k):
    resultado = identidad(len(m))
    while k:
        if k & 1:
            resultado = producto(resultado, m)
        m = producto(m, m)
        k >>= 1
    return resultado


a = [[1, 2, 3], [4, 5, 6]]
print("Transpuesta:", transpuesta(a))
print("Rotada a la derecha:", rota_horario(a))
print("Rotada a la izquierda:", rota_antihorario(a))
try:
    producto(a, a)
except ValueError as e:
    print("Error:", e)

Transpuesta: [[1, 4], [2, 5], [3, 6]]
Rotada a la derecha: [[4, 1], [5, 2], [6, 3]]
Rotada a la izquierda: [[3, 6], [2, 5], [1, 4]]
Error: no se puede multiplicar 2×3 por 2×3


In [2]:
rng = np.random.default_rng(2026)
for _ in range(100):
    f, c, k = rng.integers(1, 6, size=3)
    m = rng.integers(-5, 6, size=(f, c))
    n = rng.integers(-5, 6, size=(c, k))
    cuadrada = rng.integers(-3, 4, size=(f, f))
    exponente = int(rng.integers(0, 8))
    assert np.array_equal(transpuesta(m.tolist()), m.T)
    assert np.array_equal(producto(m.tolist(), n.tolist()), m @ n)
    assert np.array_equal(rota_horario(m.tolist()), np.rot90(m, -1))
    assert np.array_equal(rota_antihorario(m.tolist()), np.rot90(m))
    assert np.array_equal(potencia(cuadrada.tolist(), exponente), np.linalg.matrix_power(cuadrada, exponente))

FIB = [[1, 1], [1, 0]]
exacto = potencia(FIB, 100)[0][1]
con_int64 = np.linalg.matrix_power(np.array(FIB, dtype=np.int64), 100)[0][1]
con_object = np.linalg.matrix_power(np.array(FIB, dtype=object), 100)[0][1]
print(f"F(100) con listas:      {exacto}")
print(f"F(100) con NumPy int64: {con_int64}")
print(f"F(100) con dtype=object: {con_object}")
a_mano, b_mano = 0, 1
for _ in range(100):
    a_mano, b_mano = b_mano, a_mano + b_mano
assert exacto == a_mano == con_object != con_int64

F(100) con listas:      354224848179261915075
F(100) con NumPy int64: 3736710778780434371
F(100) con dtype=object: 354224848179261915075


## 2. Recorrer y generar una espiral

Escribir `espiral(m)`, que recorra una matriz de $m \times n$ en espiral en sentido horario desde la esquina superior izquierda, moviendo cuatro límites (arriba, abajo, izquierda y derecha) hacia dentro. Comprobarla contra una segunda versión que *pela* la matriz: toma la primera fila y gira el resto 90° a la izquierda, hasta vaciarla. Escribir también `matriz_espiral(filas, columnas)`, que rellene una matriz con $1, 2, 3, \dots$ en ese mismo orden girando cuando el siguiente paso se sale o cae en una casilla ocupada, y comprobar la ida y vuelta: recorrer en espiral la matriz espiral da $1, 2, \dots, mn$.

In [3]:
def espiral(m):
    if not m:
        return []
    resultado = []
    arriba, abajo, izquierda, derecha = 0, len(m) - 1, 0, len(m[0]) - 1
    while arriba <= abajo and izquierda <= derecha:
        resultado += m[arriba][izquierda:derecha + 1]
        resultado += [m[f][derecha] for f in range(arriba + 1, abajo + 1)]
        if arriba < abajo:
            resultado += m[abajo][izquierda:derecha][::-1]
        if izquierda < derecha:
            resultado += [m[f][izquierda] for f in range(abajo - 1, arriba, -1)]
        arriba, abajo, izquierda, derecha = arriba + 1, abajo - 1, izquierda + 1, derecha - 1
    return resultado


def espiral_pelando(m):
    m, resultado = [list(fila) for fila in m], []
    while m:
        resultado += m.pop(0)
        m = rota_antihorario(m) if m else []
    return resultado


def matriz_espiral(filas, columnas):
    m = [[0] * columnas for _ in range(filas)]
    f, c, d = 0, 0, 0
    pasos = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    for k in range(1, filas * columnas + 1):
        m[f][c] = k
        nf, nc = f + pasos[d][0], c + pasos[d][1]
        if not (0 <= nf < filas and 0 <= nc < columnas) or m[nf][nc]:
            d = (d + 1) % 4
            nf, nc = f + pasos[d][0], c + pasos[d][1]
        f, c = nf, nc
    return m


for fila in matriz_espiral(5, 5):
    print(" ".join(f"{x:2}" for x in fila))
print()
for fila in matriz_espiral(3, 6):
    print(" ".join(f"{x:2}" for x in fila))
print(espiral([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))

for filas, columnas in product(range(1, 9), repeat=2):
    assert espiral(matriz_espiral(filas, columnas)) == list(range(1, filas * columnas + 1))
rng = random.Random(3)
for _ in range(300):
    columnas = rng.randint(1, 7)
    m = [[rng.randint(0, 99) for _ in range(columnas)] for _ in range(rng.randint(1, 7))]
    assert espiral(m) == espiral_pelando(m)

 1  2  3  4  5
16 17 18 19  6
15 24 25 20  7
14 23 22 21  8
13 12 11 10  9

 1  2  3  4  5  6
14 15 16 17 18  7
13 12 11 10  9  8
[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


## 3. ¿Es un cuadrado mágico?

Un cuadrado mágico normal de orden $n$ contiene los números de $1$ a $n^2$ una vez cada uno y todas sus filas, columnas y las dos diagonales suman la constante mágica $n(n^2+1)/2$. Escribir `es_magico(m)` con listas y `es_magico_np(m)` con NumPy (`sum(axis=...)`, `trace` y `fliplr`) y compararlas en tableros aleatorios. Construir después cuadrados mágicos de orden impar con el método siamés (empezar en el centro de la primera fila y avanzar en diagonal hacia arriba y a la derecha dando la vuelta por los bordes; si la casilla está ocupada, bajar una) y validarlos hasta orden 15. Comprobar también el cuadrado de Durero de orden 4.

In [4]:
def es_magico(m):
    n = len(m)
    if any(len(fila) != n for fila in m) or sorted(chain(*m)) != list(range(1, n * n + 1)):
        return False
    lineas = [*m, *zip(*m), [m[i][i] for i in range(n)], [m[i][n - 1 - i] for i in range(n)]]
    return all(sum(linea) == n * (n * n + 1) // 2 for linea in lineas)


def es_magico_np(m):
    a = np.array(m)
    n = len(a)
    if a.shape != (n, n) or not np.array_equal(np.sort(a, axis=None), np.arange(1, n * n + 1)):
        return False
    sumas = np.concatenate([a.sum(axis=0), a.sum(axis=1), [np.trace(a), np.trace(np.fliplr(a))]])
    return bool((sumas == n * (n * n + 1) // 2).all())


def siames(n):
    if n % 2 == 0:
        raise ValueError("el método siamés solo sirve para orden impar")
    m = [[0] * n for _ in range(n)]
    f, c = 0, n // 2
    for k in range(1, n * n + 1):
        m[f][c] = k
        nf, nc = (f - 1) % n, (c + 1) % n
        if m[nf][nc]:
            nf, nc = (f + 1) % n, c
        f, c = nf, nc
    return m


for fila in siames(5):
    print(" ".join(f"{x:2}" for x in fila))
DURERO = [[16, 3, 2, 13], [5, 10, 11, 8], [9, 6, 7, 12], [4, 15, 14, 1]]
print("Durero:", es_magico(DURERO))

assert siames(3) == [[8, 1, 6], [3, 5, 7], [4, 9, 2]]
assert es_magico(DURERO) and es_magico_np(DURERO)
for n in range(1, 16, 2):
    assert es_magico(siames(n)) and es_magico_np(siames(n))
rng = random.Random(15)
for _ in range(3000):
    n = rng.choice([3, 4])
    numeros = rng.sample(range(1, n * n + 1), n * n)
    m = [numeros[i:i + n] for i in range(0, n * n, n)]
    assert es_magico(m) == es_magico_np(m)
mal = [fila[:] for fila in siames(5)]
mal[0][0], mal[0][1] = mal[0][1], mal[0][0]
assert not es_magico(mal) and not es_magico([[1, 2], [3, 4]]) and not es_magico([[1, 1], [1, 1]])

17 24  1  8 15
23  5  7 14 16
 4  6 13 20 22
10 12 19 21  3
11 18 25  2  9
Durero: True


## 4. ¿Es un sudoku resuelto válido?

Escribir `sudoku_valido(t)`, que diga si un tablero de 9 × 9 está bien resuelto: cada fila, cada columna y cada caja de 3 × 3 contiene los dígitos del 1 al 9. Escribir una segunda versión con NumPy que obtenga las cajas con `reshape(3, 3, 3, 3).swapaxes(1, 2)`. Generar sudokus resueltos aleatorios a partir de un patrón fijo aplicando transformaciones que conservan la validez (permutar dígitos, filas dentro de una banda, bandas, columnas dentro de una pila, pilas y trasponer) y comprobar que los dos validadores los aceptan y rechazan los que tienen una casilla cambiada o dos casillas de una fila intercambiadas.

In [5]:
DIGITOS = set(range(1, 10))


def cajas(t):
    return ([t[f][c] for f in range(F, F + 3) for c in range(C, C + 3)] for F in (0, 3, 6) for C in (0, 3, 6))


def sudoku_valido(t):
    if len(t) != 9 or any(len(fila) != 9 for fila in t):
        return False
    return all(set(grupo) == DIGITOS for grupo in chain(t, zip(*t), cajas(t)))


def sudoku_valido_np(t):
    a = np.array(t)
    if a.shape != (9, 9):
        return False
    grupos = np.concatenate([a, a.T, a.reshape(3, 3, 3, 3).swapaxes(1, 2).reshape(9, 9)])
    return bool((np.sort(grupos, axis=1) == np.arange(1, 10)).all())


def sudoku_aleatorio(rng):
    def mezcla_grupos():
        bandas = rng.sample(range(3), 3)
        return [3 * b + i for b in bandas for i in rng.sample(range(3), 3)]
    digitos = rng.sample(range(1, 10), 9)
    filas, columnas = mezcla_grupos(), mezcla_grupos()
    t = [[digitos[(3 * (f % 3) + f // 3 + c) % 9] for c in columnas] for f in filas]
    return transpuesta(t) if rng.random() < 0.5 else t


rng = random.Random(2026)
ejemplo = sudoku_aleatorio(rng)
for i, fila in enumerate(ejemplo):
    print(" ".join(f"{''.join(map(str, fila[j:j + 3]))}" for j in (0, 3, 6)))
    if i in (2, 5):
        print()
print("Válido:", sudoku_valido(ejemplo))

for _ in range(500):
    t = sudoku_aleatorio(rng)
    assert sudoku_valido(t) and sudoku_valido_np(t)
    f, c = rng.randrange(9), rng.randrange(9)
    cambiado = [fila[:] for fila in t]
    cambiado[f][c] = cambiado[f][c] % 9 + 1
    assert not sudoku_valido(cambiado) and not sudoku_valido_np(cambiado)
    c1, c2 = rng.sample(range(9), 2)
    cambiado = [fila[:] for fila in t]
    cambiado[f][c1], cambiado[f][c2] = cambiado[f][c2], cambiado[f][c1]
    assert not sudoku_valido(cambiado) and not sudoku_valido_np(cambiado)

723 958 614
164 372 598
589 164 273

416 237 985
958 416 732
372 895 146

895 641 327
237 589 461
641 723 859
Válido: True


## 5. Línea y bingo

Un cartón de bingo español tiene 3 filas y 9 columnas con 15 números: 5 por fila, cada columna con los de su decena (la primera del 1 al 9, la última del 80 al 90) y ordenados de menor a mayor dentro de la columna. Escribir `carton_valido(carton)` y `jugar(carton, bolas)`, que devuelva cuántas bolas hacen falta para cantar línea (completar una fila) y bingo (todo el cartón). Comprobar el resultado con la posición de cada número en el sorteo y estimar con 5000 sorteos la media de bolas necesarias.

In [6]:
CARTON = [
    [3, None, 21, None, 45, None, 62, None, 81],
    [None, 14, 27, 38, None, 56, None, 74, None],
    [8, None, None, 39, None, None, 69, 77, 90],
]


def decena(columna):
    return range(1, 10) if columna == 0 else range(80, 91) if columna == 8 else range(10 * columna, 10 * columna + 10)


def carton_valido(carton):
    if len(carton) != 3 or any(len(fila) != 9 for fila in carton):
        return False
    if any(sum(x is not None for x in fila) != 5 for fila in carton):
        return False
    for c, columna in enumerate(zip(*carton)):
        numeros = [x for x in columna if x is not None]
        if not numeros or any(x not in decena(c) for x in numeros) or numeros != sorted(set(numeros)):
            return False
    return True


def jugar(carton, bolas):
    pendientes = [{x for x in fila if x is not None} for fila in carton]
    linea = None
    for i, bola in enumerate(bolas, start=1):
        for fila in pendientes:
            fila.discard(bola)
        if linea is None and any(not fila for fila in pendientes):
            linea = i
        if all(not fila for fila in pendientes):
            return linea, i
    return linea, None


def dibuja(carton, salidas=()):
    for fila in carton:
        print(" ".join("  " if x is None else "**" if x in salidas else f"{x:2}" for x in fila))


rng = random.Random(90)
bolas = rng.sample(range(1, 91), 90)
linea, bingo = jugar(CARTON, bolas)
print(f"Línea con {linea} bolas y bingo con {bingo}")
dibuja(CARTON, set(bolas[:linea]))
print("Válido:", carton_valido(CARTON))
mal = [fila[:] for fila in CARTON]
mal[0][1] = 19
print("Con un 19 de más en la primera fila:", carton_valido(mal))

Línea con 73 bolas y bingo con 88
 3    **    **    **    **
   ** ** **    **    **   
**       **       69 ** **
Válido: True
Con un 19 de más en la primera fila: False


In [7]:
def jugar_por_posiciones(carton, bolas):
    posicion = {bola: i for i, bola in enumerate(bolas, start=1)}
    ultimas = [max(posicion[x] for x in fila if x is not None) for fila in carton]
    return min(ultimas), max(ultimas)


lineas, bingos = [], []
for _ in range(5000):
    bolas = rng.sample(range(1, 91), 90)
    resultado = jugar(CARTON, bolas)
    assert resultado == jugar_por_posiciones(CARTON, bolas)
    lineas.append(resultado[0])
    bingos.append(resultado[1])
print(f"Media de bolas: línea {np.mean(lineas):.1f}, bingo {np.mean(bingos):.1f}")
assert carton_valido(CARTON) and not carton_valido(mal)
assert min(lineas) >= 5 and min(bingos) >= 15

Media de bolas: línea 64.6, bingo 85.3


## 6. ¿Quién ha ganado al tres en raya?

Escribir `estado(tablero)`, que reciba un tablero de tres en raya como tres cadenas con `X`, `O` y `.` (empieza siempre `X`) y diga si `gana X`, `gana O`, es `empate`, sigue `en juego` o es `imposible`: hay más `O` que `X` o dos `X` más que `O`, ganan los dos, o el ganador no tiene el número de fichas de quien acaba de mover. Comprobarlo contra todas las partidas posibles: recorrer con BFS los tableros alcanzables desde el vacío (sin seguir jugando cuando alguien gana) y verificar que, de los $3^9$ tableros, `estado` declara imposibles justo los que no se alcanzan y clasifica bien los demás.

In [8]:
LINEAS = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]


def ganadores(casillas):
    return {j for j in "XO" for linea in LINEAS if all(casillas[i] == j for i in linea)}


def estado(tablero):
    casillas = "".join(tablero)
    x, o = casillas.count("X"), casillas.count("O")
    gana = ganadores(casillas)
    if x not in (o, o + 1) or len(gana) == 2:
        return "imposible"
    if "X" in gana:
        return "gana X" if x == o + 1 else "imposible"
    if "O" in gana:
        return "gana O" if x == o else "imposible"
    return "empate" if x + o == 9 else "en juego"


ejemplos = [
    ["XOX", "XOO", "OXX"],
    ["XXX", "OO.", "..."],
    ["XXX", "OOO", "..."],
    ["XO.", ".XO", "..X"],
    ["OOO", "XX.", "X.X"],
    ["X.O", ".X.", "O.."],
]
for t in ejemplos:
    print(" | ".join(t), "→", estado(t))

XOX | XOO | OXX → empate
XXX | OO. | ... → gana X
XXX | OOO | ... → imposible
XO. | .XO | ..X → gana X
OOO | XX. | X.X → imposible
X.O | .X. | O.. → en juego


In [9]:
def alcanzables():
    vistos, cola = {"." * 9}, deque(["." * 9])
    while cola:
        t = cola.popleft()
        if ganadores(t) or "." not in t:
            continue
        ficha = "X" if t.count("X") == t.count("O") else "O"
        for i, c in enumerate(t):
            if c == ".":
                nuevo = t[:i] + ficha + t[i + 1:]
                if nuevo not in vistos:
                    vistos.add(nuevo)
                    cola.append(nuevo)
    return vistos


legales = alcanzables()
todos = ["".join(p) for p in product("XO.", repeat=9)]
posibles = {t for t in todos if estado([t[:3], t[3:6], t[6:]]) != "imposible"}
print(f"{len(legales)} tableros alcanzables de {len(todos)}")
print(Counter(estado([t[:3], t[3:6], t[6:]]) for t in legales))
assert posibles == legales

5478 tableros alcanzables de 19683
Counter({'en juego': 4520, 'gana X': 626, 'gana O': 316, 'empate': 16})


## 7. Perímetro de una isla sin recorrerla

En una rejilla de `1` (tierra) y `0` (agua), el perímetro de la tierra es el número de lados de casilla que dan al agua o al borde (en `17-grafos` se calcula recorriendo cada isla con BFS). Calcularlo sin recorrer nada de tres formas: casilla a casilla mirando sus cuatro lados; con la fórmula $4 \cdot \text{tierra} - 2 \cdot \text{contactos}$, donde cada contacto es una pareja de casillas de tierra vecinas en horizontal o vertical, y con NumPy, rodeando la rejilla de agua con `np.pad` y contando los cambios con `np.diff`. Ver que un lago interior también suma costa y comparar las tres formas en rejillas aleatorias.

In [10]:
def perimetro_lados(r):
    filas, columnas = len(r), len(r[0])
    lados = 0
    for f, c in product(range(filas), range(columnas)):
        if r[f][c]:
            for df, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nf, nc = f + df, c + dc
                lados += not (0 <= nf < filas and 0 <= nc < columnas and r[nf][nc])
    return lados


def perimetro_formula(r):
    tierra = sum(map(sum, r))
    contactos = sum(a and b for fila in r for a, b in zip(fila, fila[1:]))
    contactos += sum(a and b for col in zip(*r) for a, b in zip(col, col[1:]))
    return 4 * tierra - 2 * contactos


def perimetro_np(r):
    a = np.pad(np.array(r, dtype=int), 1)
    return int(np.abs(np.diff(a, axis=0)).sum() + np.abs(np.diff(a, axis=1)).sum())


isla = [
    [0, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 0, 1, 1],
    [0, 1, 1, 1, 0],
]
anillo = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
for nombre, r in [("isla con lago", isla), ("anillo", anillo), ("casilla suelta", [[1]])]:
    print(f"{nombre:>14}: {perimetro_lados(r)}")

assert perimetro_lados(anillo) == 16 and perimetro_lados([[1]]) == 4
rng = random.Random(21)
for _ in range(500):
    filas, columnas = rng.randint(1, 10), rng.randint(1, 10)
    r = [[int(rng.random() < 0.55) for _ in range(columnas)] for _ in range(filas)]
    assert perimetro_lados(r) == perimetro_formula(r) == perimetro_np(r)

 isla con lago: 22
        anillo: 16
casilla suelta: 4


## 8. Un tablero de hundir la flota

Un tablero de 10 × 10 de hundir la flota (`#` barco, `.` agua) es válido si sus barcos son segmentos rectos horizontales o verticales que no se tocan ni por los lados ni en diagonal, y la flota es exactamente un barco de 4 casillas, dos de 3, tres de 2 y cuatro de 1. Escribir `flota_valida(tablero)`, que agrupe las casillas en componentes conexas por las 8 direcciones y exija que cada una quepa en una sola fila o columna. Escribir una segunda versión sin recorridos: la forma es correcta si ninguna ventana de 2 × 2 tiene más de dos casillas de barco ni dos en diagonal, y los barcos se cuentan con los tramos de fila y de columna (`itertools.groupby`). Generar flotas aleatorias válidas, estropearlas moviendo casillas y comparar los dos validadores.

In [11]:
FLOTA = Counter({4: 1, 3: 2, 2: 3, 1: 4})


def componentes(tablero):
    pendientes = {(f, c) for f, fila in enumerate(tablero) for c, x in enumerate(fila) if x == "#"}
    while pendientes:
        pila, grupo = [pendientes.pop()], []
        while pila:
            f, c = pila.pop()
            grupo.append((f, c))
            for df, dc in product((-1, 0, 1), repeat=2):
                if (f + df, c + dc) in pendientes:
                    pendientes.remove((f + df, c + dc))
                    pila.append((f + df, c + dc))
        yield grupo


def flota_valida(tablero):
    tamanos = Counter()
    for grupo in componentes(tablero):
        filas, columnas = {f for f, _ in grupo}, {c for _, c in grupo}
        if len(filas) > 1 and len(columnas) > 1:
            return False
        tamanos[len(grupo)] += 1
    return tamanos == FLOTA


def tramos(lineas):
    return [len(list(g)) for linea in lineas for es_barco, g in groupby(linea) if es_barco]


def flota_valida_ventanas(tablero):
    a = np.array([[x == "#" for x in fila] for fila in tablero])
    so, se, io, ie = a[:-1, :-1], a[:-1, 1:], a[1:, :-1], a[1:, 1:]
    suma = so.astype(int) + se + io + ie
    diagonal = (suma == 2) & ((so & ie) | (se & io))
    if (suma > 2).any() or diagonal.any():
        return False
    por_filas, por_columnas = tramos(a.tolist()), tramos(a.T.tolist())
    largos = [x for x in por_filas + por_columnas if x > 1]
    # cada casilla de un barco vertical es un tramo de fila de longitud 1
    sueltas = por_filas.count(1) - sum(x for x in por_columnas if x > 1)
    return Counter(largos) + Counter({1: sueltas}) == FLOTA


TABLERO = [
    "#...##....",
    "#.........",
    "#..#...###",
    "#..#......",
    "...#..#...",
    "......#..#",
    ".##.......",
    "........#.",
    "#...#.....",
    "..........",
]
print("Ejemplo:", flota_valida(TABLERO), flota_valida_ventanas(TABLERO))
assert flota_valida(TABLERO) and flota_valida_ventanas(TABLERO)
tocando = TABLERO[:]
tocando[1] = "#....#...."
print("Barcos que se tocan:", flota_valida(tocando), flota_valida_ventanas(tocando))
assert not flota_valida(tocando) and not flota_valida_ventanas(tocando)

Ejemplo: True True
Barcos que se tocan: False False


In [12]:
def flota_aleatoria(rng, n=10):
    while True:
        ocupadas, tablero = set(), [["."] * n for _ in range(n)]
        for largo in sorted(FLOTA.elements(), reverse=True):
            for _ in range(200):
                horizontal = rng.random() < 0.5
                f = rng.randrange(n - (0 if horizontal else largo - 1))
                c = rng.randrange(n - (largo - 1 if horizontal else 0))
                celdas = [(f, c + i) if horizontal else (f + i, c) for i in range(largo)]
                if not any((y + dy, x + dx) in ocupadas for y, x in celdas for dy, dx in product((-1, 0, 1), repeat=2)):
                    ocupadas.update(celdas)
                    break
            else:
                break
        else:
            for f, c in ocupadas:
                tablero[f][c] = "#"
            return ["".join(fila) for fila in tablero]


rng = random.Random(1588)
for fila in flota_aleatoria(rng):
    print(fila)
resultados = Counter()
for _ in range(1500):
    tablero = flota_aleatoria(rng)
    assert flota_valida(tablero) and flota_valida_ventanas(tablero)
    celdas = [list(fila) for fila in tablero]
    barco = rng.choice([(f, c) for f in range(10) for c in range(10) if celdas[f][c] == "#"])
    agua = rng.choice([(f, c) for f in range(10) for c in range(10) if celdas[f][c] == "."])
    celdas[barco[0]][barco[1]], celdas[agua[0]][agua[1]] = ".", "#"
    movido = ["".join(fila) for fila in celdas]
    valido = flota_valida(movido)
    assert valido == flota_valida_ventanas(movido)
    resultados[valido] += 1
print("Tableros con una casilla movida que siguen siendo válidos:", resultados[True], "de", resultados.total())

...#..#.#.
...#....#.
##.#....#.
...#......
.....#....
###......#
......#...
........##
..........
##........


Tableros con una casilla movida que siguen siendo válidos: 140 de 1500


## 9. La hormiga de Langton

Una hormiga camina por una rejilla infinita de casillas blancas: si está en una blanca, gira 90° a la derecha, la pinta de negro y avanza; si está en una negra, gira a la izquierda, la pinta de blanco y avanza. Simularla guardando las casillas negras en un conjunto y dibujar el patrón tras 500 pasos. Mostrar cada 2000 pasos cuántas casillas negras hay y a qué distancia está la hormiga del origen, para ver cuándo empieza a alejarse en línea recta (la *autopista*). Comprobar la simulación con NumPy en una rejilla finita suficientemente grande y que las reglas son reversibles: deshacer los pasos uno a uno devuelve la rejilla en blanco.

In [13]:
PASOS = [(-1, 0), (0, 1), (1, 0), (0, -1)]


def hormiga(n, negras=None, pos=(0, 0), d=0):
    negras = set() if negras is None else negras
    f, c = pos
    for _ in range(n):
        if (f, c) in negras:
            d = (d - 1) % 4
            negras.remove((f, c))
        else:
            d = (d + 1) % 4
            negras.add((f, c))
        f, c = f + PASOS[d][0], c + PASOS[d][1]
    return negras, (f, c), d


def deshacer(n, negras, pos, d):
    f, c = pos
    for _ in range(n):
        f, c = f - PASOS[d][0], c - PASOS[d][1]
        if (f, c) in negras:
            negras.remove((f, c))
            d = (d - 1) % 4
        else:
            negras.add((f, c))
            d = (d + 1) % 4
    return negras, (f, c), d


def dibuja_rejilla(negras, pos):
    filas = [f for f, _ in negras | {pos}]
    columnas = [c for _, c in negras | {pos}]
    for f in range(min(filas), max(filas) + 1):
        print("".join("H" if (f, c) == pos else "#" if (f, c) in negras else "." for c in range(min(columnas), max(columnas) + 1)))


negras, pos, d = hormiga(500)
dibuja_rejilla(negras, pos)

..##....##...
.#..#....####
###.....###.H
#.####.#.#.#.
......#...#..
...##.#.###..
..#..##...#..
..#..#..##...
....##.......
.#.....####.#
.#.##.....###
..##....#..#.
...##....##..


In [14]:
print(f"{'pasos':>6} {'negras':>7} {'distancia':>10}")
negras, pos, d = set(), (0, 0), 0
for pasos in range(2000, 14001, 2000):
    negras, pos, d = hormiga(2000, negras, pos, d)
    print(f"{pasos:>6} {len(negras):>7} {abs(pos[0]) + abs(pos[1]):>10}")


def hormiga_np(n, lado=201):
    rejilla = np.zeros((lado, lado), dtype=bool)
    f = c = lado // 2
    d = 0
    for _ in range(n):
        d = (d - 1) % 4 if rejilla[f, c] else (d + 1) % 4
        rejilla[f, c] = ~rejilla[f, c]
        f, c = f + PASOS[d][0], c + PASOS[d][1]
    return {(int(y) - lado // 2, int(x) - lado // 2) for y, x in np.argwhere(rejilla)}


assert hormiga_np(11000) == hormiga(11000)[0]
for n in [1, 50, 500, 12000]:
    assert deshacer(n, *hormiga(n)) == (set(), (0, 0), 0)

 pasos  negras  distancia
  2000     202         12
  4000     306         30
  6000     442          2
  8000     562         14
 10000     720         26
 12000     952         88
 14000    1182        166


## 10. Distancias en una rejilla hexagonal

En una rejilla hexagonal con coordenadas axiales $(q, r)$ cada casilla tiene seis vecinas, y con la tercera coordenada cúbica $s = -q - r$ la distancia entre dos casillas es $\max(|\Delta q|, |\Delta r|, |\Delta s|)$. Escribir `distancia_hex(a, b)`, dibujar como texto un hexágono de radio 3 con la distancia de cada casilla al centro y comprobar que a distancia $k$ hay $6k$ casillas. Los tableros suelen numerar las casillas por fila y columna con las filas impares desplazadas media casilla (coordenadas *offset*), donde las vecinas dependen de la paridad de la fila: convertir esas coordenadas a axiales y comprobar la fórmula con un BFS sobre las vecinas *offset* desde varios orígenes.

In [15]:
VECINAS_AXIALES = [(1, 0), (1, -1), (0, -1), (-1, 0), (-1, 1), (0, 1)]


def distancia_hex(a, b):
    dq, dr = a[0] - b[0], a[1] - b[1]
    return max(abs(dq), abs(dr), abs(dq + dr))


def offset_a_axial(fila, columna):
    return columna - (fila - (fila & 1)) // 2, fila


def vecinas_offset(fila, columna):
    if fila % 2 == 0:
        saltos = [(0, 1), (-1, 0), (-1, -1), (0, -1), (1, -1), (1, 0)]
    else:
        saltos = [(0, 1), (-1, 1), (-1, 0), (0, -1), (1, 0), (1, 1)]
    return [(fila + df, columna + dc) for df, dc in saltos]


def bfs_offset(origen, radio):
    distancias, cola = {origen: 0}, deque([origen])
    while cola:
        actual = cola.popleft()
        if distancias[actual] == radio:
            continue
        for v in vecinas_offset(*actual):
            if v not in distancias:
                distancias[v] = distancias[actual] + 1
                cola.append(v)
    return distancias


R = 3
for r in range(-R, R + 1):
    fila = [distancia_hex((q, r), (0, 0)) for q in range(max(-R, -r - R), min(R, -r + R) + 1)]
    print(" " * abs(r) + " ".join(map(str, fila)))

alrededor = Counter(distancia_hex((q, r), (0, 0)) for q in range(-8, 9) for r in range(-8, 9))
assert all(alrededor[k] == 6 * k for k in range(1, 9))
assert all(distancia_hex((0, 0), v) == 1 for v in VECINAS_AXIALES)
for origen in [(0, 0), (3, 4), (-5, 2), (7, -3)]:
    for celda, d in bfs_offset(origen, 10).items():
        assert d == distancia_hex(offset_a_axial(*origen), offset_a_axial(*celda))
print("Del (0, 0) al (6, 6) en offset:", distancia_hex(offset_a_axial(0, 0), offset_a_axial(6, 6)))

   3 3 3 3
  3 2 2 2 3
 3 2 1 1 2 3
3 2 1 0 1 2 3
 3 2 1 1 2 3
  3 2 2 2 3
   3 3 3 3
Del (0, 0) al (6, 6) en offset: 9


## 11. Colisión de asteroides

Una fila de asteroides avanza por una línea: el valor absoluto es su tamaño y el signo, su sentido (positivo hacia la derecha, negativo hacia la izquierda). Cuando dos chocan, el pequeño se destruye y, si son iguales, los dos; los que van en el mismo sentido nunca se encuentran. Escribir `colisiones(asteroides)`, que devuelva los que quedan, con una pila en una sola pasada: cada asteroide que va a la izquierda destruye las cimas menores que van a la derecha. Comprobarla contra una simulación ingenua que resuelve el primer choque (un positivo seguido de un negativo) una y otra vez hasta que no hay más.

In [16]:
def colisiones(asteroides):
    pila = []
    for a in asteroides:
        while a < 0 and pila and pila[-1] > 0:
            if pila[-1] < -a:
                pila.pop()
                continue
            if pila[-1] == -a:
                pila.pop()
            a = 0
        if a:
            pila.append(a)
    return pila


def colisiones_ingenuo(asteroides):
    fila = list(asteroides)
    while (i := next((i for i in range(len(fila) - 1) if fila[i] > 0 > fila[i + 1]), None)) is not None:
        derecha, izquierda = fila[i], -fila[i + 1]
        quedan = [fila[i]] if derecha > izquierda else [fila[i + 1]] if izquierda > derecha else []
        fila[i:i + 2] = quedan
    return fila


for fila in [[5, 10, -5], [8, -8], [10, 2, -5], [-2, -1, 1, 2], [3, 4, -6, 1, -1, 7]]:
    print(f"{str(fila):<24} → {colisiones(fila)}")

rng = random.Random(42)
for _ in range(5000):
    fila = [rng.choice([-1, 1]) * rng.randint(1, 5) for _ in range(rng.randint(0, 12))]
    assert colisiones(fila) == colisiones_ingenuo(fila)

[5, 10, -5]              → [5, 10]
[8, -8]                  → []
[10, 2, -5]              → [10]
[-2, -1, 1, 2]           → [-2, -1, 1, 2]
[3, 4, -6, 1, -1, 7]     → [-6, 7]


## 12. Puntuar una tirada de cinco dados

En los juegos de cinco dados tipo Yahtzee una tirada puede apuntarse en una categoría: unos a seises (suma de los dados de ese valor), trío y póker (suma de todos si hay al menos 3 o 4 iguales), full (25 puntos: un trío y una pareja), escalera corta (30: cuatro valores consecutivos), escalera larga (40: cinco consecutivos), repóker (50: cinco iguales) y libre (suma de todos). Escribir `puntuaciones(dados)` con `Counter` y `mejor_categoria(dados)`. Recorrer las $6^5 = 7776$ tiradas, contar en cuántas puntúa cada categoría y comprobar los recuentos de repóker, póker, trío, full y escalera larga con combinatoria, y la escalera corta con una segunda versión que busca subcadenas en los valores ordenados.

In [17]:
NOMBRES = ["unos", "doses", "treses", "cuatros", "cincos", "seises"]


def puntuaciones(dados):
    cuenta = Counter(dados)
    valores, total, maximo = set(dados), sum(dados), max(cuenta.values())
    p = {nombre: v * cuenta[v] for v, nombre in enumerate(NOMBRES, start=1)}
    p["trío"] = total if maximo >= 3 else 0
    p["póker"] = total if maximo >= 4 else 0
    p["full"] = 25 if sorted(cuenta.values()) == [2, 3] else 0
    p["escalera corta"] = 30 if any(set(range(i, i + 4)) <= valores for i in (1, 2, 3)) else 0
    p["escalera larga"] = 40 if valores in ({1, 2, 3, 4, 5}, {2, 3, 4, 5, 6}) else 0
    p["repóker"] = 50 if maximo == 5 else 0
    p["libre"] = total
    return p


def mejor_categoria(dados):
    return max(puntuaciones(dados).items(), key=lambda par: par[1])


rng = random.Random(6)
for _ in range(6):
    dados = sorted(rng.choices(range(1, 7), k=5))
    nombre, puntos = mejor_categoria(dados)
    print(f"{dados}: {nombre} ({puntos})")
print("[4, 4, 4, 4, 4]: {} ({})".format(*mejor_categoria([4, 4, 4, 4, 4])))

[1, 2, 3, 5, 5]: libre (16)
[3, 3, 4, 5, 5]: libre (20)
[2, 3, 4, 5, 5]: escalera corta (30)
[2, 2, 4, 5, 5]: libre (18)
[1, 3, 5, 5, 6]: libre (20)
[1, 2, 3, 5, 5]: libre (16)
[4, 4, 4, 4, 4]: repóker (50)


In [18]:
tiradas = list(product(range(1, 7), repeat=5))
puntuan = Counter(nombre for t in tiradas for nombre, puntos in puntuaciones(t).items() if puntos)
for nombre in ["trío", "póker", "full", "escalera corta", "escalera larga", "repóker"]:
    print(f"{nombre:>15}: {puntuan[nombre]:5} tiradas ({puntuan[nombre] / len(tiradas):6.2%})")

exactamente = {k: 6 * comb(5, k) * 5 ** (5 - k) for k in (3, 4)}
assert puntuan["repóker"] == 6
assert puntuan["póker"] == exactamente[4] + 6
assert puntuan["trío"] == exactamente[3] + exactamente[4] + 6
assert puntuan["full"] == 6 * 5 * comb(5, 3)
assert puntuan["escalera larga"] == 2 * 120
corta = sum(any(s in "".join(map(str, sorted(set(t)))) for s in ("1234", "2345", "3456")) for t in tiradas)
assert puntuan["escalera corta"] == corta

           trío:  1656 tiradas (21.30%)
          póker:   156 tiradas ( 2.01%)
           full:   300 tiradas ( 3.86%)
 escalera corta:  1200 tiradas (15.43%)
 escalera larga:   240 tiradas ( 3.09%)
        repóker:     6 tiradas ( 0.08%)


## 13. Un intercambio justo

Dos amigos tienen bolsas de cromos con distintos valores y quieren intercambiar un cromo cada uno para que los dos tengan el mismo valor total. Si las sumas son $S_a$ y $S_b$ y se cambian $x$ de la primera por $y$ de la segunda, hace falta $S_a - x + y = S_b - y + x$, es decir, $y = x - (S_a - S_b)/2$. Escribir `intercambio_justo(a, b)`, que devuelva una pareja válida (o `None`) en $O(n + m)$ guardando la segunda bolsa en un conjunto, y `todos_los_intercambios(a, b)`. Compararlas con la fuerza bruta que prueba todas las parejas.

In [19]:
def todos_los_intercambios(a, b):
    diferencia = sum(a) - sum(b)
    if diferencia % 2:
        return set()
    en_b = set(b)
    return {(x, x - diferencia // 2) for x in a if x - diferencia // 2 in en_b}


def intercambio_justo(a, b):
    diferencia = sum(a) - sum(b)
    if diferencia % 2:
        return None
    en_b = set(b)
    return next(((x, x - diferencia // 2) for x in a if x - diferencia // 2 in en_b), None)


def intercambios_bruta(a, b):
    return {(x, y) for x in a for y in b if sum(a) - x + y == sum(b) - y + x}


ana, bruno = [5, 8, 2, 10], [7, 1, 4, 3, 6]
print(f"Ana suma {sum(ana)} y Bruno {sum(bruno)}")
x, y = intercambio_justo(ana, bruno)
print(f"Ana da {x} y Bruno da {y}: los dos quedan con {sum(ana) - x + y}")
print("Todos los posibles:", sorted(todos_los_intercambios(ana, bruno)))
print("Con [1, 2] y [4]:", intercambio_justo([1, 2], [4]))

rng = random.Random(10)
for _ in range(3000):
    a = [rng.randint(1, 12) for _ in range(rng.randint(1, 6))]
    b = [rng.randint(1, 12) for _ in range(rng.randint(1, 6))]
    bruta = intercambios_bruta(a, b)
    assert todos_los_intercambios(a, b) == bruta
    pareja = intercambio_justo(a, b)
    assert (pareja is None) == (not bruta) and (pareja is None or pareja in bruta)

Ana suma 25 y Bruno 21
Ana da 5 y Bruno da 3: los dos quedan con 23
Todos los posibles: [(5, 3), (8, 6)]
Con [1, 2] y [4]: None
